# Cap do 1 — Knowledge Tracing tren do thi (Kaggle GPU)

Notebook **clone repo** [Knowledge-Tracing](https://github.com/linhnnh688/Knowledge-Tracing) roi goi ham trong `level1/`.

| Yeu cau de | Cach lam |
|---|---|
| Dataset cong khai | ASSISTments (`skill_builder_data.csv` / smoke: `assistment_test15.csv`) |
| Baseline khong do thi | **DKT** (`GKT/models.py`) |
| Mo hinh do thi | **GKT** (PyTorch) + **GIKT** port PyTorch |
| Ablation | GKT `Transition` vs `Identity` vs `Zero` |
| Giai thich co ban | 5 case dung + 5 case sai, lich su + lang gieng |

Ban goc `GIKT/` la TensorFlow 1.12 (`tf.contrib`) — khong chay tren Kaggle. Logic cap 1 nam o source, notebook chi orchestrate.

## 0. Clone repo va cau hinh

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/linhnnh688/Knowledge-Tracing.git"
KAGGLE_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / "Knowledge-Tracing"
    if not REPO.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
    else:
        subprocess.call(["git", "-C", str(REPO), "pull", "--ff-only"])
else:
    REPO = Path.cwd() if (Path.cwd() / "level1").exists() else Path.cwd() / "Knowledge-Tracing"

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo:", REPO.resolve())

In [ ]:
# Kaggle da co PyTorch GPU — khong pip install torch.
%pip install -q pandas scipy scikit-learn matplotlib tqdm

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from level1.pipeline import run_level1
from level1.explain import collect_cases, format_cases, skill_lookup_from_loader

print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

QUICK_RUN = True  # False de train skill_builder_data.csv (ASSISTments 2009)
DATA_FILE = "assistment_test15.csv" if QUICK_RUN else "skill_builder_data.csv"
EPOCHS = 8 if QUICK_RUN else 20
BATCH_SIZE = 16 if QUICK_RUN else 32
MAX_SEQ_LEN = 80 if QUICK_RUN else 200
SAVE_DIR = Path("/kaggle/working/level1_out") if IS_KAGGLE else (REPO / "outputs" / "level1")
print(DATA_FILE, "epochs", EPOCHS)

## 1. Review ngan GKT vs GIKT

**GKT** (Nakagawa et al., WI 2019) — `GKT/`
- Dinh: knowledge concept; moi concept co hidden state `h_c`.
- Buoc t: aggregate embedding cua concept vua lam + GNN theo ma tran ke (`Dense` / `Transition` / ...).
- Erase-Add gate + GRUCell cap nhat toan bo concept, roi `P(dung | concept tiep theo)`.
- DKT nam trong cung file: LSTM tren `(skill, response)`, khong do thi.

**GIKT** (Yang et al., ECML-PKDD 2020) — `GIKT/` (TF1) va `level1/gikt.py` (PyTorch)
- Do thi luong phan **cau hoi – skill**; SumAggregator lan embedding n-hop.
- LSTM tren embedding cau hoi da lan truyen + cau tra loi; tuong tac voi cau hoi ke tiep.
- Source goc can `tensorflow==1.12` / `tf.contrib` nen cap 1 dung ban PyTorch, cung split user voi GKT.

## 2. EDA

In [ ]:
df = pd.read_csv(REPO / "GKT" / "data" / DATA_FILE)
print(df.shape, "users", df["user_id"].nunique(), "skills", df["skill_id"].nunique())
print("correct rate", float(df["correct"].eq(1).mean()))
user_len = df.groupby("user_id").size()
ax = user_len.hist(bins=40, figsize=(8, 3))
ax.set_title("Interactions per user")
ax.set_xlabel("length")
plt.show()

## 3. Train cap do 1

`run_level1` (trong `level1/pipeline.py`) se:
1. Split **theo user** (cung seed), luu `split.json`
2. Do thi Transition **chi tu tap train**
3. Train DKT, GKT-Transition, GKT-Identity, GKT-Zero, GIKT, GIKT-empty
4. Bao cao AUC / Accuracy / NLL tren test

In [ ]:
level1_out = run_level1(
    data_file=DATA_FILE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    hid_dim=32,
    emb_dim=32,
    seed=42,
    max_seq_len=MAX_SEQ_LEN,
    run_gikt=True,
    save_dir=SAVE_DIR,
)
table = pd.DataFrame(level1_out["table"])
display(table)

## 4. Bang so sanh va ablation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for r in level1_out["results"]:
    axes[0].plot(r["history"]["val_loss"], label=r["name"])
    axes[1].plot(r["history"]["val_auc"], label=r["name"])
axes[0].set_title("Val NLL")
axes[0].set_xlabel("epoch")
axes[1].set_title("Val AUC")
axes[1].set_xlabel("epoch")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(table))
ax.bar([i - 0.2 for i in x], table["test_auc"], width=0.4, label="AUC")
ax.bar([i + 0.2 for i in x], table["test_acc"], width=0.4, label="Acc")
ax.set_xticks(list(x))
ax.set_xticklabels(table["model"], rotation=20)
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Test AUC / Accuracy")
plt.tight_layout()
plt.show()

gkt_rows = table[table["model"].str.startswith("GKT")]
print("Ablation GKT (Transition vs Identity vs Zero):")
print(gkt_rows.to_string(index=False))

## 5. 5 case dung + 5 case sai

Moi case: ky nang muc tieu, xac suat, lich su lam bai, canh hang xom tren do thi Transition (train).

In [ ]:
gkt = next(r for r in level1_out["results"] if r["name"] == "GKT-Transition")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = gkt["model"].to(device)
bundle = gkt["bundle"]
lookup = skill_lookup_from_loader(bundle["test_loader"])
graph = bundle.get("graph")
if graph is not None:
    graph = graph.detach().cpu()

buckets = collect_cases(
    model,
    bundle["test_loader"],
    graph=graph,
    skill_lookup=lookup,
    n_each=5,
    history_k=8,
    device=device,
    model_type="GKT",
)
report = format_cases(buckets)
print(report)
(SAVE_DIR / "cases.md").write_text(report, encoding="utf-8")
print("saved", SAVE_DIR / "cases.md")

## 6. Luu ket qua

In [ ]:
out_csv = Path("/kaggle/working/level1_metrics.csv") if IS_KAGGLE else (SAVE_DIR / "metrics.csv")
table.to_csv(out_csv, index=False)
print("metrics:", out_csv)
print("split:", SAVE_DIR / "split.json")
print("ckpts:", list(Path(SAVE_DIR).glob("*.pt")))
table